# Large datasets (Plan 03): build and train within Colab's memory

This notebook checks that the larger RelBench datasets fit in Colab's **12.7 GB** of system RAM with `--features-on-disk text`. That option:
- reads the raw tables one at a time;
- keeps the text vectors on disk, memory-mapped.

Each dataset is downloaded only when its turn comes. Then three steps run, each in a fresh process (`experiments/benchmarks/graph_memory.py`), and each records its peak memory:

| Step | What it does |
|---|---|
| `ours` | builds the graph from the raw tables, one table at a time, including the GloVe text features |
| `ours-cached` | loads the graph from the cache, as every later run does |
| `train` | `train.py`'s steps: load the graph, train `EPOCHS` epoch(s) on the GPU, evaluate on validation |

RelBench's own path is not run here: `get_db()` alone needs 9.8 GB for `rel-event`, measured before any features are built, and running out of memory can restart the Colab runtime.

**On Colab:**
1. *Runtime → Change runtime type → GPU*.
2. *Runtime → Run all*.
3. If the runtime restarts (out of memory), the rows saved so far are in Drive. The failed step shows `failed (exit -9)` once it is rerun, or is missing.
4. The last cell downloads `results.csv`. Upload it, and this notebook saved with its outputs (*File → Download → .ipynb*).


In [1]:
# ---- Settings ----
PROJECT_REPO = 'https://github.com/HARSHALK2598/cs224w-structural-optimizer'
PROJECT_BRANCH = 'step/03-2-tables-one-at-a-time'  # train.py, disk_features.py, graph_memory.py
PYG_REPO = 'https://github.com/HARSHALK2598/pytorch_geometric'
PYG_BRANCH = 'contrib/structural-optimizer'
RELBENCH_COMMIT = '91c1f6420a03feb61c279aaeb194d7268435f278'  # as in scripts/setup_env.sh

# (dataset, task), smallest database first. Each is downloaded when its turn comes.
DATASETS = [
    ('rel-event', 'user-repeat'),
    ('rel-hm', 'user-churn'),
    ('rel-avito', 'user-visits'),
    ('rel-stack', 'user-engagement'),
]
TEXT_EMBEDDER = 'glove'
EPOCHS = 1
STEPS = ['ours', 'ours-cached', 'train']
DELETE_AFTER = True      # free the disk after each dataset (download and cache)
USE_DRIVE = True         # Colab only: keep results in Google Drive
DRIVE_DIR = '/content/drive/MyDrive/cs224w-structural-optimizer'


## 1. Environment

The same setup as `colab_sweep.ipynb`:
- **On Colab:** clones both repositories and installs pinned versions.
- **Elsewhere:** uses the current environment and repository.


In [2]:
import os, subprocess, sys, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules


def sh(cmd, cwd=None):
    """Runs a shell command, streaming its output; stops on failure."""
    print('$', cmd, flush=True)
    # PYTHONUNBUFFERED: show the sweep's lines as they happen, not at the end.
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'command failed ({proc.returncode}): {cmd}')


HAS_GPU = shutil.which('nvidia-smi') is not None and subprocess.run(
    ['nvidia-smi'], capture_output=True).returncode == 0

if IN_COLAB:
    ROOT = Path('/content/cs224w-structural-optimizer')
    if not ROOT.exists():
        sh(f'git clone -q --branch {PROJECT_BRANCH} {PROJECT_REPO} {ROOT}')
    else:
        sh(f'git -C {ROOT} pull -q origin {PROJECT_BRANCH}')
    pyg = ROOT / 'pytorch_geometric'
    if not pyg.exists():
        sh(f'git clone -q --branch {PYG_BRANCH} {PYG_REPO} {pyg}')
    else:
        sh(f'git -C {pyg} pull -q origin {PYG_BRANCH}')

    build = 'cu126' if HAS_GPU else 'cpu'
    pip = f'{sys.executable} -m pip install -q'
    sh(f'{sys.executable} -m pip uninstall -q -y torchvision torchaudio')
    sh(f'{pip} torch==2.12.0 --index-url https://download.pytorch.org/whl/{build}')
    sh(f'{pip} pyg_lib==0.9.0 -f https://data.pyg.org/whl/torch-2.12.0+{build}.html')
    sh(f'{pip} -e {pyg}')
    sh(f'{pip} "relbench @ git+https://github.com/snap-stanford/relbench@{RELBENCH_COMMIT}" '
       f'pytorch-frame==0.3.0 sentence-transformers==6.2.0 pyyaml pandas')
else:
    # Run from anywhere inside the project repository (or set PROJECT_ROOT).
    ROOT = Path(os.environ.get('PROJECT_ROOT', Path.cwd())).resolve()
    while not (ROOT / 'experiments' / 'sweep.py').exists():
        if ROOT.parent == ROOT:
            raise RuntimeError('Start the notebook inside the project repository.')
        ROOT = ROOT.parent

os.chdir(ROOT)
print('project root:', ROOT, '| Colab:', IN_COLAB, '| GPU:', HAS_GPU)
sh(f'{sys.executable} -c "import torch, torch_geometric, torch_frame, relbench, pyg_lib; '
   f'print(\'torch\', torch.__version__, \'| cuda\', torch.cuda.is_available(), '
   f'torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'\', '
   f'\'| pyg\', torch_geometric.__version__, \'| frame\', torch_frame.__version__, '
   f'\'| relbench\', relbench.__version__, \'| pyg_lib\', pyg_lib.__version__)"')
sh('git log --oneline -1')
sh('git -C pytorch_geometric log --oneline -1' if (ROOT / 'pytorch_geometric').exists() else 'true')

$ git clone -q --branch step/03-2-tables-one-at-a-time https://github.com/HARSHALK2598/cs224w-structural-optimizer /content/cs224w-structural-optimizer
$ git clone -q --branch contrib/structural-optimizer https://github.com/HARSHALK2598/pytorch_geometric /content/cs224w-structural-optimizer/pytorch_geometric
$ /usr/bin/python3 -m pip uninstall -q -y torchvision torchaudio
$ /usr/bin/python3 -m pip install -q torch==2.12.0 --index-url https://download.pytorch.org/whl/cpu
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.30 requires torchvision, which is not installed.
fastai 2.8.12 requires torchvision>=0.11, which is not installed.
$ /usr/bin/python3 -m pip install -q pyg_lib==0.9.0 -f https://data.pyg.org/whl/torch-2.12.0+cpu.html
$ /usr/bin/python3 -m pip install -q -e /content/cs224w-structural-optimizer/pytorch_geometric
$ /usr/bin/python3 -m pip ins

## 2. Where results go

One row per (dataset, step) is added to `graph_memory/results.csv`. On Colab with `USE_DRIVE = True`, that file is in your Google Drive, so it survives a restart.

| Column | Meaning |
|---|---|
| `peak_gb` | the step's peak system RAM |
| `gpu_peak_gb` | the step's peak GPU memory (`train` only) |
| `minutes` | how long the step took |
| `val` | the validation score after training (`train` only) |
| `cache_files` | feature files already cached when the step started; 0 for a first build |


In [4]:
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path(DRIVE_DIR) / 'graph_memory' / 'results.csv'
else:
    OUT = ROOT / 'experiments' / 'results' / 'graph_memory' / 'results.csv'
OUT.parent.mkdir(parents=True, exist_ok=True)
print('results file:', OUT)
sh('free -g; df -h /; nvidia-smi --query-gpu=name,memory.total --format=csv || true')


Mounted at /content/drive
results file: /content/drive/MyDrive/cs224w-structural-optimizer/graph_memory/results.csv
$ free -g; df -h /; nvidia-smi --query-gpu=name,memory.total --format=csv || true
               total        used        free      shared  buff/cache   available
Mem:              12           1           7           0           4          11
Swap:              0           0           0
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G  1.1G  107G   2% /
/bin/sh: 1: nvidia-smi: not found


## 3. One dataset at a time

For each dataset, the cell:
1. downloads it (database and task labels; nothing else);
2. runs the steps, printing one JSON line per step and the training log;
3. if `DELETE_AFTER` is set, deletes that dataset's download and cache.

A dataset that fails does not stop the others.


In [5]:
import json
HF_DATASETS = ROOT / 'data' / 'hf' / 'hub' / 'datasets--stanford-star--relbench-v1'

for dataset, task in DATASETS:
    print(f'\n======== {dataset} / {task} ========', flush=True)
    try:
        sh(f'{sys.executable} -c "import os; os.environ.setdefault(\'HF_HOME\', \'{ROOT}/data/hf\'); '
           f'from relbench import load_dataset; d = load_dataset(\'{dataset}\'); print(d.dataset_dir)"')
        sh(f'du -shL {HF_DATASETS}/snapshots/*/{dataset}/db || true')
        sh(f'{sys.executable} experiments/benchmarks/graph_memory.py {dataset} --task {task} '
           f'--text-embedder {TEXT_EMBEDDER} --epochs {EPOCHS} --cache-dir train '
           f'--steps {" ".join(STEPS)} --out "{OUT}"')
    except RuntimeError as error:
        print('FAILED:', error, flush=True)
    if DELETE_AFTER:  # Only this dataset's files: its links and the files they point to.
        for folder in HF_DATASETS.glob(f'snapshots/*/{dataset}'):
            for link in folder.rglob('*'):
                if link.is_symlink():
                    link.resolve().unlink(missing_ok=True)
            shutil.rmtree(folder, ignore_errors=True)
        shutil.rmtree(ROOT / 'data' / 'materialized' / dataset, ignore_errors=True)



======== rel-event / user-repeat ========
$ /usr/bin/python3 -c "import os; os.environ.setdefault('HF_HOME', '/content/cs224w-structural-optimizer/data/hf'); from relbench import load_dataset; d = load_dataset('rel-event'); print(d.dataset_dir)"

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 20 files: 100%|██████████| 20/20 [00:02<00:00,  9.48it/s]
/content/cs224w-structural-optimizer/data/hf/hub/datasets--stanford-star--relbench-v1/snapshots/21d7284df2a9621c71f21bc4a1a30b7e8effb8dd/rel-event
$ du -shL /content/cs224w-structural-optimizer/data/hf/hub/datasets--stanford-star--relbench-v1/snapshots/*/rel-event/db || true
114M	/content/cs224w-structural-optimizer/data/hf/hub/datasets--stanford-star--relbench-v1/snapshots/21d7284df2a9621c71f21bc4a1a30b7e8effb8dd/rel-event/db
$ /usr/bin/python3 experiments/benchmarks/graph_memory.py 

KeyboardInterrupt: 

## 4. Results

Steps whose `peak_gb` is under 12.7 fit in Colab's memory.


In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
res = pd.read_csv(OUT)
print(res.to_string(index=False))


## 5. Download the results

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(OUT))
